# Exact IP sanity check

Run the exact influence-propagation IP on the same PA graph batches used by the exact script, then verify that the seed set it returns actually induces full spread.

In [1]:
from __future__ import annotations

import itertools
import sys
import time
from pathlib import Path

import gurobipy as gp
import networkx as nx
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from technology_diffusion.helpers import connected_component_spread
from technology_diffusion.ip_problems import build_exact_ip

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)

In [2]:
def theta_const(g: nx.Graph, value: int) -> dict[int, int]:
    return {node: value for node in g.nodes()}


def theta_from_degrees(g: nx.Graph, rule: str = 'simple') -> dict[int, int]:
    if rule == 'simple':
        return {node: max(1, min(2, g.degree(node))) for node in g.nodes()}
    raise ValueError(f'Unknown rule: {rule}')


DEFAULT_C_LIST = [1, 2, 5]
DEFAULT_N_LIST = [6, 8, 10, 12, 15]
DEFAULT_SEED_LIST = [1, 42, 53, 99, 101]
DEFAULT_INIT_NODES = 5
DEFAULT_INIT_MODE = 'complete'


def build_cases() -> list[dict]:
    from technology_diffusion.helpers import create_pa_graph

    cases = []
    for n_nodes, c, seed in itertools.product(DEFAULT_N_LIST, DEFAULT_C_LIST, DEFAULT_SEED_LIST):
        g, thetas = create_pa_graph(
            n_nodes=n_nodes,
            c=c,
            seed=seed,
            init_nodes=DEFAULT_INIT_NODES,
            init_mode=DEFAULT_INIT_MODE,
        )
        cases.append(
            {
                'name': f'pa_n{n_nodes}_c{c}_seed{seed}',
                'n': n_nodes,
                'm': g.number_of_edges(),
                'c': c,
                'seed': seed,
                'init_nodes': DEFAULT_INIT_NODES,
                'init_mode': DEFAULT_INIT_MODE,
                'graph': g,
                'thetas': thetas,
            }
        )
    return cases


def solve_exact_case(case: dict, max_time: float = 60.0) -> dict:
    g: nx.Graph = case['graph']
    thetas: dict[int, int] = case['thetas']
    n_nodes = g.number_of_nodes()

    env = gp.Env(empty=True)
    env.setParam('OutputFlag', 0)
    env.setParam('LogToConsole', 0)
    env.start()

    build_start = time.perf_counter()
    model, x, y, nvar = build_exact_ip(
        g,
        thetas,
        k=n_nodes,
        use_simultaneous=True,
        time_horizon=n_nodes,
        max_time=max_time,
        env=env,
    )
    build_time = time.perf_counter() - build_start

    if model is None:
        return {
            'name': case['name'],
            'n': n_nodes,
            'm': g.number_of_edges(),
            'c': case['c'],
            'seed': case['seed'],
            'status': 'build_failed',
            'build_time_s': round(build_time, 4),
            'solve_time_s': None,
            'objective_seeds': None,
            'seed_nodes': None,
            'full_spread': False,
            'spread': None,
            'history': None,
        }

    model.setParam('OutputFlag', 0)
    model.setParam('LogToConsole', 0)
    model.setParam('TimeLimit', max_time - build_time)
    model.optimize()

    solve_time = float(model.Runtime)
    total_time = build_time + solve_time

    seed_nodes = [node for node in g.nodes() if x[node, 0].X > 0.5]
    seed_vector = np.zeros(n_nodes, dtype=float)
    for node in seed_nodes:
        seed_vector[node] = 1.0

    spread, spread_hist, final_x = connected_component_spread(g, seed_vector, thetas, max_t=1000)
    objective_seeds = int(round(model.objVal)) if model.SolCount > 0 else None
    full_spread = spread == n_nodes

    return {
        'name': case['name'],
        'n': n_nodes,
        'm': g.number_of_edges(),
        'c': case['c'],
        'seed': case['seed'],
        'status': 'solved' if model.SolCount > 0 else 'no_solution',
        'build_time_s': round(build_time, 4),
        'solve_time_s': round(solve_time, 4),
        'total_time_s': round(total_time, 4),
        'objective_seeds': objective_seeds,
        'seed_nodes': seed_nodes,
        'spread': spread,
        'full_spread': full_spread,
        'spread_history': spread_hist,
    }

In [3]:
cases = build_cases()
results = []
for index, case in enumerate(cases, start=1):
    print(f"[{index}/{len(cases)}] running {case['name']} (n={case['n']}, c={case['c']}, seed={case['seed']})")
    result = solve_exact_case(case, max_time=1200.0)
    results.append(result)
    print(f"[{index}/{len(cases)}] done {case['name']} -> spread={result['spread']} full_spread={result['full_spread']} seeds={result['objective_seeds']}")

summary = pd.DataFrame(results)
summary[['name', 'n', 'm', 'status', 'objective_seeds', 'seed_nodes', 'spread', 'full_spread', 'build_time_s', 'solve_time_s']]

[1/75] running pa_n6_c1_seed1 (n=6, c=1, seed=1)
[1/75] done pa_n6_c1_seed1 -> spread=6 full_spread=True seeds=1
[2/75] running pa_n6_c1_seed42 (n=6, c=1, seed=42)
[2/75] done pa_n6_c1_seed42 -> spread=6 full_spread=True seeds=2
[3/75] running pa_n6_c1_seed53 (n=6, c=1, seed=53)
[3/75] done pa_n6_c1_seed53 -> spread=6 full_spread=True seeds=2
[4/75] running pa_n6_c1_seed99 (n=6, c=1, seed=99)
[4/75] done pa_n6_c1_seed99 -> spread=6 full_spread=True seeds=2
[5/75] running pa_n6_c1_seed101 (n=6, c=1, seed=101)
[5/75] done pa_n6_c1_seed101 -> spread=6 full_spread=True seeds=1
[6/75] running pa_n6_c2_seed1 (n=6, c=2, seed=1)
[6/75] done pa_n6_c2_seed1 -> spread=6 full_spread=True seeds=1
[7/75] running pa_n6_c2_seed42 (n=6, c=2, seed=42)
[7/75] done pa_n6_c2_seed42 -> spread=6 full_spread=True seeds=2
[8/75] running pa_n6_c2_seed53 (n=6, c=2, seed=53)
[8/75] done pa_n6_c2_seed53 -> spread=6 full_spread=True seeds=3
[9/75] running pa_n6_c2_seed99 (n=6, c=2, seed=99)
[9/75] done pa_n6_c2_see

,name,n,m,status,objective_seeds,seed_nodes,spread,full_spread,build_time_s,solve_time_s
0,pa_n6_c1_seed1,6,12,solved,1,[0],6,True,0.0152,0.0308
1,pa_n6_c1_seed42,6,11,solved,2,"[2, 5]",6,True,0.0111,0.1377
2,pa_n6_c1_seed53,6,14,solved,2,"[3, 4]",6,True,0.0117,0.1822
3,pa_n6_c1_seed99,6,14,solved,2,"[0, 2]",6,True,0.0116,0.0947
4,pa_n6_c1_seed101,6,12,solved,1,[2],6,True,0.0119,0.0198
...,...,...,...,...,...,...,...,...,...,...
70,pa_n15_c5_seed1,15,35,solved,5,"[2, 3, 6, 7, 10]",15,True,0.6214,20.4190
71,pa_n15_c5_seed42,15,36,solved,6,"[1, 5, 7, 12, 13, 14]",15,True,0.6499,18.5461
72,pa_n15_c5_seed53,15,36,solved,6,"[0, 2, 4, 7, 12, 13]",15,True,0.7040,23.4455
73,pa_n15_c5_seed99,15,37,solved,6,"[0, 2, 6, 7, 8, 14]",15,True,0.7027,634.7649


In [7]:
all_good = bool(summary['full_spread'].all())
full_spread_ok = bool(summary['full_spread'].all())

print(f'All cases full spread: {full_spread_ok}')
print(f'Overall sanity check passed: {all_good}')

summary[['name', 'objective_seeds', 'seed_nodes', 'spread', 'full_spread']]

All cases full spread: False
Overall sanity check passed: False


,name,objective_seeds,seed_nodes,spread,full_spread
0,pa_n6_c1_seed1,1,[0],6,True
1,pa_n6_c1_seed42,2,"[2, 5]",6,True
2,pa_n6_c1_seed53,2,"[3, 4]",6,True
3,pa_n6_c1_seed99,2,"[0, 2]",6,True
4,pa_n6_c1_seed101,1,[2],6,True
...,...,...,...,...,...
70,pa_n15_c5_seed1,5,"[2, 3, 6, 7, 10]",15,True
71,pa_n15_c5_seed42,6,"[1, 5, 7, 12, 13, 14]",15,True
72,pa_n15_c5_seed53,6,"[0, 2, 4, 7, 12, 13]",15,True
73,pa_n15_c5_seed99,6,"[0, 2, 6, 7, 8, 14]",15,True


In [8]:
# Inspect one critical instance with solver output enabled.
# Edit the parameters below to target the graph you want to inspect.
critical_n = 12
critical_c = 1
critical_seed = 99
critical_init_nodes = DEFAULT_INIT_NODES
critical_init_mode = DEFAULT_INIT_MODE
critical_max_time = 60.0

from technology_diffusion.helpers import create_pa_graph

critical_graph, critical_thetas = create_pa_graph(
    n_nodes=critical_n,
    c=critical_c,
    seed=critical_seed,
    init_nodes=critical_init_nodes,
    init_mode=critical_init_mode,
)

critical_env = gp.Env(empty=True)
critical_env.setParam('OutputFlag', 1)
critical_env.setParam('LogToConsole', 1)
critical_env.start()

critical_start = time.perf_counter()
critical_model, critical_x, critical_y, critical_nvar = build_exact_ip(
    critical_graph,
    critical_thetas,
    k=critical_n,
    use_simultaneous=True,
    time_horizon=None,
    max_time=critical_max_time,
    env=critical_env,
)
critical_build_time = time.perf_counter() - critical_start

print(f'Critical instance: n={critical_n}, c={critical_c}, seed={critical_seed}')
print(f'Build time: {critical_build_time:.4f}s')

if critical_model is None:
    print('Exact IP build failed for the selected instance.')
else:
    critical_model.setParam('OutputFlag', 1)
    critical_model.setParam('LogToConsole', 1)
    critical_model.setParam('TimeLimit', max(0.0, critical_max_time - critical_build_time))
    critical_model.optimize()

    critical_seed_nodes = [node for node in critical_graph.nodes() if critical_x[node, 0].X > 0.5]
    critical_seed_vector = np.zeros(critical_n, dtype=float)
    for node in critical_seed_nodes:
        critical_seed_vector[node] = 1.0

    critical_spread, critical_history, critical_final_x = connected_component_spread(
        critical_graph,
        critical_seed_vector,
        critical_thetas,
        max_t=1000,
    )

    print(f'Status: {critical_model.Status}')
    print(f'Objective seeds: {int(round(critical_model.objVal)) if critical_model.SolCount > 0 else None}')
    print(f'Seed nodes: {critical_seed_nodes}')
    print(f'Full spread: {critical_spread == critical_n}')
    print(f'Spread history: {critical_history}')

Set parameter Username
Set parameter LicenseID to value 2777966
Academic license - for non-commercial use only - expires 2027-02-11
Critical instance: n=12, c=1, seed=99
Build time: 0.2222s
Set parameter OutputFlag to value 1
Set parameter LogToConsole to value 1
Set parameter TimeLimit to value 5.9777804124998511e+01
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.5.0 25F71)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  5.9777804124998511e+01

Optimize a model with 7393 rows, 3756 columns and 63096 nonzeros
Model fingerprint: 0xfcfb987d
Variable types: 0 continuous, 3756 integer (2028 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+01]
Found heuristic solution: objective 11.0000000
Presolve removed 433 rows and 0 columns
Presolve time: 0.04s
Presolved: 6960 

In [ ]:
# Print all cases where full_spread is False
failed = summary[~summary['full_spread']]
print(f'Failing cases: {len(failed)}')
failed[['name','n','c','seed','status','objective_seeds','seed_nodes','spread','full_spread']]